<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 6. Сбор и импорт данных — подробная методичка

Эта методичка продолжает Этапы 1–5. Предполагается, что у вас:
- `db` и `app` в `healthy`, `/health` возвращает 200;
- JWT-аутентификация работает (`/auth/register`, `/auth/login`, `/auth/me`);
- мультиагентная система (Supervisor + Validator + Critic + Editor) реализована и покрыта тестами;
- таблицы `glossaries` и `translation_memories` созданы (Этап 3), но пусты.

Если чего-то нет — вернитесь к Этапу 5.

**Цель этапа:** наполнить БД эталонными данными — **≥ 1000 пар TM** и **≥ 500 терминов глоссария** — и подготовить eval-датасеты для Этапа 12.

**Недель:** 1.

**Что вы получите в конце этапа:**

- `scripts/eval/prepare_eval_datasets.py` — скачивание и подготовка 4 пар (ar↔ru, ru↔tt, tg↔fa, ru↔ba) с HuggingFace.
- `scripts/import/add_glossary.py` — импорт терминов из CSV (два формата) или builtin-наборов.
- `scripts/import/add_tm.py` — импорт TM из CSV, JSON или builtin-наборов.
- `data/import/README.md` — источники и лицензии.
- Данные в БД, проверенные через SQL.

**Критерии готовности:**

- [ ] `python scripts/eval/prepare_eval_datasets.py` создаёт `eval_datasets/<pair>_<size>.json` и `manifest.json`.
- [ ] `python scripts/import/add_glossary.py --csv <file>` добавляет термины; повторный запуск → `added=0`.
- [ ] `python scripts/import/add_tm.py --json <file>` добавляет TM-пары; повторный запуск → `added=0`.
- [ ] `SELECT COUNT(*) FROM translation_memories` ≥ 1000 (или ≥ 300 для промежуточной сдачи).
- [ ] `SELECT COUNT(*) FROM glossaries` ≥ 500 (или ≥ 150 для промежуточной сдачи).
- [ ] `data/import/README.md` с источниками и лицензиями.
- [ ] Ветка `week-6` создана, коммит сделан.

> **Важно.** Этот этап **не трогает** `docker-compose.yml`, `main.py` и модели. Все скрипты — отдельные entry-point'ы, которые используют `glossary_manager` и `tm_manager` из `backend/data/`. Qdrant ещё не подключён (это Этап 7) — при импорте TM будут предупреждения `Qdrant unavailable; skipping vector addition` в логах. Это нормально: BM25-индекс обновится в памяти, а Qdrant заполнится в Этапе 7 после полной переиндексации.

---

## 6.0. Подготовка: ветка и зависимости

```powershell
cd D:\RUNG
git checkout -b week-6
```

Проверьте, что в `pyproject.toml` есть `datasets = "^5.0.1"`. Это библиотека HuggingFace для `load_dataset`. Если её нет:

```powershell
python -m poetry add datasets
```

Проверьте, что есть `HF_TOKEN`. В `infrastructure/.env` (Этап 1) уже прописан `HF_TOKEN=hf_...`. Скрипты Этапа 6 читают `HF_TOKEN` через `os.getenv("HF_TOKEN")` напрямую, **не через `Settings`** — значит, `.env` автоматически не подхватывается. Нужно экспортировать:

**Windows PowerShell:**
```powershell
$env:HF_TOKEN = "hf_..."
```

**Linux/macOS:**
```bash
export HF_TOKEN=hf_...
```

> **Почему не через `Settings`.** `prepare_eval_datasets.py` — отдельный скрипт, может запускаться без `backend.core.config` (например, на машине без остальных зависимостей). Прямое чтение `os.getenv` делает его standalone. Плюс `load_dataset(..., token=...)` — это API `datasets`, а не Pydantic — `Settings.HF_TOKEN` пришлось бы передавать отдельным аргументом.

---

## 6.1. `scripts/eval/prepare_eval_datasets.py` — подготовка eval-датасетов

**Что это:** скрипт скачивает 4 параллельных корпуса с HuggingFace, нормализует (убирает пустые/слишком длинные), балансирует по доменам и сохраняет в JSON-формат, который понимает `scripts/eval/run_eval.py` (Этап 12).

**Зачем это нужно:** eval-датасет — это эталонные пары `source → reference` фиксированного размера (10, 20, 50, ..., 1000), на которых сравниваются 8 ablation-режимов. Без него невозможны ни `run_eval.py`, ни `paired_bootstrap.py`, ни `report_tables.py`. Один раз сгенерировали — используем во всех прогонах.

Создайте `scripts/eval/prepare_eval_datasets.py`:

```python
#!/usr/bin/env python3
"""
Prepare evaluation datasets in the format consumed by scripts/eval/run_eval.py.

Usage:
    export HF_TOKEN=hf_...
    python scripts/eval/prepare_eval_datasets.py

Optional env:
    HF_TOKEN              Hugging Face token (required for gated datasets)
    EVAL_OUTPUT_DIR       output directory (default: ./eval_datasets)
    EVAL_SEED             sampling seed (default: 42)
    EVAL_SIZES            comma-separated sizes (default: 10,20,50,100,200,500,1000)
    EVAL_PAIRS            comma-separated pair names (default: all)
    EVAL_MAX_SRC_LEN      max length of source text; 0 = no limit (default: 600)
    EVAL_MAX_TGT_LEN      max length of reference text; 0 = no limit (default: 0)
    EVAL_NEWS_MIN_KEEP    min number of "news" examples to keep even if long
                          (shortest ones are picked). 0 = disable. (default: 3)
    EVAL_NEWS_ABS_MAX_LEN hard cap for forced news picks; 0 = no cap
                          (default: 2000)

Output JSON schema:
    {
        "sources":     [...],
        "references":  [...],
        "domains":     [...],
        "source_lang": "ar",
        "target_lang": "ru",
        "metadata":    {...}
    }
"""

import json
import os
import random
import sys
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple

try:
    from datasets import load_dataset
except ImportError:
    print("Install datasets: pip install datasets", file=sys.stderr)
    sys.exit(1)

# ----------------------------- CONFIG --------------------------------------

def _env_int_list(name: str, default: List[int]) -> List[int]:
    raw = os.getenv(name)
    if not raw:
        return default
    return [int(x.strip()) for x in raw.split(",") if x.strip()]


def _env_str_list(name: str, default: Optional[List[str]]) -> Optional[List[str]]:
    raw = os.getenv(name)
    if not raw:
        return default
    return [x.strip() for x in raw.split(",") if x.strip()]


def _env_int(name: str, default: int) -> int:
    raw = os.getenv(name)
    if raw is None or raw == "":
        return default
    return int(raw)


HF_TOKEN       = os.getenv("HF_TOKEN") or None
OUTPUT_DIR     = Path(os.getenv("EVAL_OUTPUT_DIR", "./eval_datasets"))
RANDOM_SEED    = _env_int("EVAL_SEED", 42)
SAMPLE_SIZES   = _env_int_list("EVAL_SIZES", [10, 20, 50, 100, 200, 500, 1000])
PAIRS_FILTER   = _env_str_list("EVAL_PAIRS", None)

MAX_SRC_LEN       = _env_int("EVAL_MAX_SRC_LEN", 600)      # 0 = no limit
MAX_TGT_LEN       = _env_int("EVAL_MAX_TGT_LEN", 0)        # 0 = no limit
NEWS_MIN_KEEP     = _env_int("EVAL_NEWS_MIN_KEEP", 3)      # 0 = disable
NEWS_ABS_MAX_LEN  = _env_int("EVAL_NEWS_ABS_MAX_LEN", 2000)

CONFIGS: List[Dict] = [
    {
        "name": "ar_ru",
        "hf_dataset": "ArabicNLPWorld/arabic-russian-parallel-corpus",
        "split": "train",
        "source_lang": "ar",
        "target_lang": "ru",
        "source_col": "arabic",
        "target_col": "russian",
        "domain_col": "source",
        "domain_map": {
            "quran":  "scripture",
            "bible":  "scripture",
            "hadith": "hadith",
            "news":   "news",
            "wiki":   "encyclopedic",
            "books":  "prose",
            "poetry": "poetry",
        },
        "default_domain": "prose",
    },
    {
        "name": "ru_tt",
        "hf_dataset": "TatarNLPWorld/tatar-english-russian-corpus",
        "split": "full",
        "source_lang": "ru",
        "target_lang": "tt",
        "source_col": "russian_text",
        "target_col": "tatar_text",
        "domain_col": None,
        "domain_map": {},
        "default_domain": "prose",
    },
    {
        "name": "tg_fa",
        "hf_dataset": "TajikNLPWorld/TajPersParallelCorpusFull",
        "split": "train",
        "source_lang": "tg",
        "target_lang": "fa",
        "source_col": "tajik",
        "target_col": "farsi",
        "domain_col": "category",
        "domain_map": {},
        "default_domain": "prose",
    },
    {
        "name": "ru_ba",
        "hf_dataset": "BorisTM/bak-rus-parallel",
        "split": "train",
        "source_lang": "ru",
        "target_lang": "ba",
        "source_col": "rus",
        "target_col": "bak",
        "domain_col": None,
        "domain_map": {},
        "default_domain": "prose",
    },
]

# --------------------------- DOMAIN HELPERS --------------------------------

_DOMAIN_ALIASES = {
    "quran":     "scripture",
    "koran":     "scripture",
    "bible":     "scripture",
    "gospel":    "scripture",
    "torah":     "scripture",
    "hadith":    "hadith",
    "sunnah":    "hadith",
    "poem":      "poetry",
    "poetry":    "poetry",
    "verse":     "poetry",
    "news":      "news",
    "article":   "news",
    "wiki":      "encyclopedic",
    "wikipedia": "encyclopedic",
    "legal":     "legal",
    "law":       "legal",
    "medical":   "medical",
    "tech":      "technical",
    "it":        "technical",
}


def normalize_domain(raw: Optional[str], default: str) -> str:
    if not raw:
        return default
    key = str(raw).strip().lower()
    if key in _DOMAIN_ALIASES:
        return _DOMAIN_ALIASES[key]
    for alias, canonical in _DOMAIN_ALIASES.items():
        if alias in key:
            return canonical
    return default


def detect_column(ds, candidates: List[str]) -> Optional[str]:
    for col in candidates:
        if col in ds.column_names:
            return col
    return None


# ------------------------------- LOADING -----------------------------------

def load_split(cfg: Dict):
    kwargs = {"split": cfg["split"]}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    ds = load_dataset(cfg["hf_dataset"], **kwargs)

    src_col = detect_column(ds, [cfg["source_col"], "source", "src"])
    tgt_col = detect_column(ds, [cfg["target_col"], "target", "tgt", "ref"])
    if src_col is None or tgt_col is None:
        raise RuntimeError(
            f"[{cfg['name']}] cannot detect columns; "
            f"available: {ds.column_names}"
        )

    domain_col = None
    if cfg.get("domain_col"):
        domain_col = detect_column(ds, [cfg["domain_col"]])

    return ds, src_col, tgt_col, domain_col


def _passes_length_filter(l_src: int, l_tgt: int) -> bool:
    if MAX_SRC_LEN and l_src > MAX_SRC_LEN:
        return False
    if MAX_TGT_LEN and l_tgt > MAX_TGT_LEN:
        return False
    return True


def extract_pairs(
    ds,
    src_col: str,
    tgt_col: str,
    domain_col: Optional[str],
    cfg: Dict,
) -> Dict:
    default_domain = cfg.get("default_domain", "prose")
    domain_map = cfg.get("domain_map", {}) or {}

    n_raw = len(ds)
    n_empty = 0

    all_rows: List[Tuple[str, str, str, int, int]] = []

    for row in ds:
        src = (row.get(src_col) or "").strip()
        tgt = (row.get(tgt_col) or "").strip()
        if not src or not tgt:
            n_empty += 1
            continue

        if domain_col is not None:
            raw_domain = row.get(domain_col)
            if raw_domain and raw_domain in domain_map:
                domain = domain_map[raw_domain]
            else:
                domain = normalize_domain(raw_domain, default_domain)
        else:
            domain = default_domain

        all_rows.append((src, tgt, domain, len(src), len(tgt)))

    news_rows  = [r for r in all_rows if r[2] == "news"]
    other_rows = [r for r in all_rows if r[2] != "news"]

    other_kept = [r for r in other_rows if _passes_length_filter(r[3], r[4])]
    news_kept  = [r for r in news_rows  if _passes_length_filter(r[3], r[4])]

    n_filtered_other = len(other_rows) - len(other_kept)
    n_filtered_news  = len(news_rows) - len(news_kept)

    news_forced = 0
    news_forced_skipped = 0
    if NEWS_MIN_KEEP > 0 and news_rows and len(news_kept) < NEWS_MIN_KEEP:
        candidates = sorted(news_rows, key=lambda r: min(r[3], r[4]))
        already = set(id(r) for r in news_kept)
        for r in candidates:
            if len(news_kept) >= NEWS_MIN_KEEP:
                break
            if id(r) in already:
                continue
            if NEWS_ABS_MAX_LEN and (r[3] > NEWS_ABS_MAX_LEN or r[4] > NEWS_ABS_MAX_LEN):
                news_forced_skipped += 1
                continue
            news_kept.append(r)
            news_forced += 1

    kept = other_kept + news_kept

    sources    = [r[0] for r in kept]
    references = [r[1] for r in kept]
    domains    = [r[2] for r in kept]

    print(f"    kept {len(sources):,} / {n_raw:,} "
          f"(empty={n_empty:,}, filtered_by_len={n_filtered_other + n_filtered_news:,})")
    if MAX_SRC_LEN or MAX_TGT_LEN:
        print(f"    length filter: src<={MAX_SRC_LEN or 'inf'}, "
              f"tgt<={MAX_TGT_LEN or 'inf'}")
    if news_rows:
        print(f"    news: {len(news_rows):,} raw -> {len(news_kept):,} kept "
              f"(forced={news_forced}, skipped_by_cap={news_forced_skipped}, "
              f"min_keep={NEWS_MIN_KEEP}, abs_max={NEWS_ABS_MAX_LEN or 'inf'})")
    print(f"    domains: {dict(Counter(domains).most_common())}")

    stats = {
        "n_raw":                n_raw,
        "n_empty":              n_empty,
        "n_filtered_by_len":    n_filtered_other + n_filtered_news,
        "length_filter": {
            "max_src_len": MAX_SRC_LEN or None,
            "max_tgt_len": MAX_TGT_LEN or None,
        },
        "news": {
            "raw":              len(news_rows),
            "kept_after_filter": len(news_kept) - news_forced,
            "forced_shortest":  news_forced,
            "skipped_by_cap":   news_forced_skipped,
            "min_keep":         NEWS_MIN_KEEP,
            "abs_max_len":      NEWS_ABS_MAX_LEN or None,
        },
    }

    return {
        "sources":    sources,
        "references": references,
        "domains":    domains,
        "stats":      stats,
    }


# ---------------------------- SAMPLING -------------------------------------

def make_sample(data: Dict, size: int, seed: int) -> Dict:
    total = len(data["sources"])
    if size >= total:
        return {
            "sources":    list(data["sources"]),
            "references": list(data["references"]),
            "domains":    list(data["domains"]),
        }

    rng = random.Random(seed)

    by_domain: Dict[str, List[int]] = {}
    for idx, dom in enumerate(data["domains"]):
        by_domain.setdefault(dom, []).append(idx)

    for idxs in by_domain.values():
        rng.shuffle(idxs)

    domains_sorted = sorted(by_domain.keys(), key=lambda d: -len(by_domain[d]))
    n_domains = len(domains_sorted)

    quota = {d: 0 for d in domains_sorted}
    remaining = size
    for d in domains_sorted:
        take = min(len(by_domain[d]), max(1, size // n_domains))
        quota[d] = take
        remaining -= take

    while remaining > 0:
        progressed = False
        for d in domains_sorted:
            if remaining == 0:
                break
            if quota[d] < len(by_domain[d]):
                quota[d] += 1
                remaining -= 1
                progressed = True
        if not progressed:
            break

    selected: List[int] = []
    for d in domains_sorted:
        selected.extend(by_domain[d][: quota[d]])
    rng.shuffle(selected)

    return {
        "sources":    [data["sources"][i] for i in selected],
        "references": [data["references"][i] for i in selected],
        "domains":    [data["domains"][i] for i in selected],
    }


# ----------------------------- WRITING -------------------------------------

def save_json(path: Path, payload: Dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def process_config(cfg: Dict, out_dir: Path) -> Dict:
    name = cfg["name"]
    print("=" * 70)
    print(f"[{name}] loading {cfg['hf_dataset']} (split={cfg['split']})")
    print("=" * 70)

    ds, src_col, tgt_col, domain_col = load_split(cfg)
    print(f"  total rows: {len(ds):,}")
    print(f"  src_col={src_col!r}, tgt_col={tgt_col!r}, domain_col={domain_col!r}")

    data = extract_pairs(ds, src_col, tgt_col, domain_col, cfg)
    stats = data["stats"]

    pair_dir = out_dir / name
    pair_dir.mkdir(parents=True, exist_ok=True)

    full_payload = {
        "sources":     data["sources"],
        "references":  data["references"],
        "domains":     data["domains"],
        "source_lang": cfg["source_lang"],
        "target_lang": cfg["target_lang"],
        "metadata": {
            "name":          name,
            "hf_dataset":    cfg["hf_dataset"],
            "split":         cfg["split"],
            "total_pairs":   len(data["sources"]),
            "domain_counts": dict(Counter(data["domains"])),
            "seed":          RANDOM_SEED,
            "sample_size":   "full",
            "stats":         stats,
        },
    }
    full_path = pair_dir / f"{name}_full.json"
    save_json(full_path, full_payload)
    print(f"  wrote {full_path} ({len(data['sources']):,} pairs)")

    sample_meta = {}
    for size in SAMPLE_SIZES:
        sample = make_sample(data, size, RANDOM_SEED)
        payload = {
            "sources":     sample["sources"],
            "references":  sample["references"],
            "domains":     sample["domains"],
            "source_lang": cfg["source_lang"],
            "target_lang": cfg["target_lang"],
            "metadata": {
                "name":          name,
                "hf_dataset":    cfg["hf_dataset"],
                "split":         cfg["split"],
                "total_pairs":   len(data["sources"]),
                "domain_counts": dict(Counter(sample["domains"])),
                "seed":          RANDOM_SEED,
                "sample_size":   size,
                "stats":         stats,
            },
        }
        sample_path = pair_dir / f"{name}_{size}.json"
        save_json(sample_path, payload)
        sample_meta[str(size)] = {
            "file":          str(sample_path.relative_to(out_dir)),
            "num_examples":  len(sample["sources"]),
            "domain_counts": dict(Counter(sample["domains"])),
        }
        print(
            f"  wrote {sample_path.name}: n={len(sample['sources'])}, "
            f"domains={dict(Counter(sample['domains']))}"
        )

    return {
        "name":          name,
        "source_lang":   cfg["source_lang"],
        "target_lang":   cfg["target_lang"],
        "hf_dataset":    cfg["hf_dataset"],
        "split":         cfg["split"],
        "total_pairs":   len(data["sources"]),
        "domain_counts": dict(Counter(data["domains"])),
        "full_file":     str(full_path.relative_to(out_dir)),
        "samples":       sample_meta,
        "stats":         stats,
    }


# -------------------------------- MAIN -------------------------------------

def main() -> None:
    if not HF_TOKEN:
        print("WARNING: HF_TOKEN is not set. Gated datasets will fail.")

    print(f"Length filter: src<={MAX_SRC_LEN or 'inf'}, "
          f"tgt<={MAX_TGT_LEN or 'inf'}")
    print(f"News min keep: {NEWS_MIN_KEEP} "
          f"(abs max len: {NEWS_ABS_MAX_LEN or 'inf'})")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary = []

    configs = CONFIGS
    if PAIRS_FILTER:
        configs = [c for c in CONFIGS if c["name"] in PAIRS_FILTER]
        print(f"Filtering pairs: {[c['name'] for c in configs]}")

    for cfg in configs:
        try:
            info = process_config(cfg, OUTPUT_DIR)
            summary.append(info)
        except Exception as e:
            print(f"[{cfg['name']}] FAILED: {e}")
            summary.append({"name": cfg["name"], "error": str(e)})

    manifest = {
        "sample_sizes": SAMPLE_SIZES,
        "seed":         RANDOM_SEED,
        "length_filter": {
            "max_src_len": MAX_SRC_LEN or None,
            "max_tgt_len": MAX_TGT_LEN or None,
        },
        "news_policy": {
            "min_keep":    NEWS_MIN_KEEP,
            "abs_max_len": NEWS_ABS_MAX_LEN or None,
        },
        "pairs": summary,
    }
    manifest_path = OUTPUT_DIR / "manifest.json"
    save_json(manifest_path, manifest)
    print("=" * 70)
    print(f"wrote {manifest_path}")
    print("=" * 70)

    for info in summary:
        if "error" in info:
            print(f"  {info['name']}: FAILED - {info['error']}")
        else:
            print(
                f"  {info['name']}: {info['total_pairs']:,} pairs, "
                f"{len(info['samples'])} samples, "
                f"domains={info['domain_counts']}"
            )


if __name__ == "__main__":
    main()
```

### Разбор ключевых решений

**Почему `load_dataset(..., split=...)`, а не `load_dataset(..., split="train").then(...)`.** `split` передаётся напрямую в API `datasets` — библиотека сама загружает только нужную часть. Для `ru_tt` split называется `full` (не `train`), для остальных — `train`. Передаём как есть из `CONFIGS`.

**Почему `detect_column` с fallback-кандидатами (`source`, `src`, `target`, `tgt`, `ref`).** Разные HF-датасеты называют колонки по-разному. Если `"arabic"` из `CONFIGS` не найдётся, `detect_column` попробует `"source"`, `"src"`. Если ничего не подходит — `RuntimeError` с `available: {column_names}`, чтобы сразу видеть, что искать.

**Почему `_DOMAIN_ALIASES` и `normalize_domain`.** HF-датасеты возвращают домены по-разному (`quran`, `koran`, `bible` — все «scripture»; `news`, `article` — оба «news»). `normalize_domain` приводит к каноническим 8: `scripture`, `hadith`, `poetry`, `news`, `encyclopedic`, `legal`, `medical`, `technical`, `prose`. Это нужно для двух вещей:
1. **Балансировки по доменам** в `make_sample`;
2. **Per-domain метрик** в `run_eval.py` (Этап 12) — он группирует по `domains` и считает chrF++/BLEU отдельно по каждой.

**Почему `domain_map` per-config, а `_DOMAIN_ALIASES` глобальная.** `domain_map` — жёсткие пары «значение из датасета → канон» (например, `quran → scripture`), специфичны для конкретного датасета. `_DOMAIN_ALIASES` — общие алиасы, применяются через `normalize_domain`, если `domain_map` не сработал.

**Почему `EVAL_MAX_SRC_LEN = 600` по умолчанию, а `EVAL_MAX_TGT_LEN = 0`.** Длинные источники (глава книги) не влезают в контекст LLM — `max_length=1024` в токенизаторе обрезает половину. 600 символов — безопасный предел для агентского pipeline. Reference может быть короче (перевод обычно компактнее) — лимит 0 значит «не ограничивать».

**Почему `EVAL_NEWS_MIN_KEEP = 3` и `EVAL_NEWS_ABS_MAX_LEN = 2000`.** Новостные тексты часто длинные, и `MAX_SRC_LEN=600` их вырезает. Но `news` — важный домен (real-world text), без него метрика нерепрезентативна. Логика: даже если все новости длинные, «форсированно» оставить `NEWS_MIN_KEEP` самых коротких, но не длиннее `NEWS_ABS_MAX_LEN=2000`. Это даёт 3 news-примера на датасет, чтобы `per_domain` не был пустым.

**Почему `make_sample` — стратифицированный, а не `random.sample`.** `random.sample` из 5000 примеров, где 90% `prose` и 5% `scripture`, даст пропорцию 90/5. Если хотите сравнить режимы **на scripture**, примеров может быть 2–3, метрика шумит. Стратифицированная выборка гарантирует минимум `size // n_domains` на домен. При 8 доменах и `size=100` — минимум 12 примеров на домен.

**Почему `size >= total → return all`.** Если `size=1000`, а в датасете 800 пар — вернуть все 800. Иначе `quota` overflow: сумма квот > `total`, `by_domain[d][:quota[d]]` вернёт меньше, чем нужно, и итог будет неполным.

**Почему `SAMPLE_SIZES = [10, 20, 50, 100, 200, 500, 1000]`.** Для ablation-экспериментов нужны разные размеры: 10/20 — smoke-тесты (быстро); 100/200 — основные прогоны; 500/1000 — финальные с узкими доверительными интервалами. Все 7 размеров пишутся за один запуск, чтобы не гонять `load_dataset` по 7 раз.

**Почему `manifest.json`.** Единый индекс: какие пары, какие размеры, какие домены. `report_tables.py` (Этап 12) читает его, чтобы знать, что есть в директории.

**Почему `save_json` с `ensure_ascii=False`.** Арабский, таджикский, татарский тексты в JSON с `ensure_ascii=True` превратятся в `\u0643\u062a...` — нечитаемо. С `False` — UTF-8 как есть.

**Почему `file=sys.stderr` в `ImportError`.** Ошибка в stderr, а не stdout — правильно для CLI. Если скрипт запускают через `python ... > output.log`, ошибка пойдёт в консоль, а не в лог.

**Почему этот скрипт **не** пишет в БД.** Это eval-датасеты, не TM. Они нужны `run_eval.py`, `paired_bootstrap.py`, `report_tables.py`. Если хотите положить их в TM (для OOD-eval), конвертируйте в `entries`-формат вручную — `add_tm.py --json` ожидает `{"source_lang", "target_lang", "entries": [...]}`, а не `{"sources", "references", ...}`.

### Запуск

**Windows PowerShell:**
```powershell
$env:HF_TOKEN = "hf_..."
python -m poetry run python scripts/eval/prepare_eval_datasets.py
```

**Linux/macOS:**
```bash
export HF_TOKEN=hf_...
python -m poetry run python scripts/eval/prepare_eval_datasets.py
```

**Что вы должны увидеть** (фрагмент для `ar_ru`):

```
======================================================================
[ar_ru] loading ArabicNLPWorld/arabic-russian-parallel-corpus (split=train)
======================================================================
  total rows: 123,456
  src_col='arabic', tgt_col='russian', domain_col='source'
    kept 45,678 / 123,456 (empty=123, filtered_by_len=77,655)
    length filter: src<=600, tgt<=inf
    news: 8,234 raw -> 3 kept (forced=3, skipped_by_cap=0, min_keep=3, abs_max=2000)
    domains: {'prose': 30,000, 'scripture': 5,000, ...}
  wrote eval_datasets/ar_ru/ar_ru_full.json (45,678 pairs)
  wrote eval_datasets/ar_ru/ar_ru_10.json: n=10, domains={'prose': 4, 'scripture': 2, ...}
  wrote eval_datasets/ar_ru/ar_ru_20.json: n=20, ...
  ...
```

### Проверка

```powershell
dir eval_datasets\ar_ru\
```

**Что вы должны увидеть** (8 файлов: full + 7 размеров):

```
ar_ru_full.json
ar_ru_10.json
ar_ru_20.json
ar_ru_50.json
ar_ru_100.json
ar_ru_200.json
ar_ru_500.json
ar_ru_1000.json
```

Проверьте один файл:

```powershell
python -m poetry run python -c "import json; d = json.load(open('eval_datasets/ar_ru/ar_ru_100.json', encoding='utf-8')); print(len(d['sources']), len(d['references']), d['source_lang'], d['target_lang']); print(d['sources'][0][:80])"
```

**Что вы должны увидеть:**

```
100 100 ar ru
<арабский текст первого примера>
```

### Коммит

```powershell
git add scripts/eval/prepare_eval_datasets.py
git commit -m "feat(week-6): add prepare_eval_datasets.py (HF download + stratified sampling)"
```

`eval_datasets/` в `.gitignore` (генерируется скриптом, не коммитится).

---

## 6.2. `scripts/import/add_glossary.py` — импорт глоссария

**Что это:** импорт терминов в таблицу `glossaries` из CSV (два формата) или builtin-наборов.

**Зачем это нужно:** глоссарий — это «словарь обязательных терминов». `Validator._check_glossary` (Этап 5) проверяет, что каждый термин из `glossary_used`, встречающийся в источнике, есть в переводе. Без глоссария `glossary_used` всегда пуст → эта проверка — no-op. Плюс `Critic` в промпте получает `Glossary terms: ...` — если пусто, LLM не знает обязательных терминов.

Создайте `scripts/import/add_glossary.py`:

```python
#!/usr/bin/env python3
"""
Add glossary entries to the database.

Sources (in priority order):
    1. --csv PATH              explicit CSV file
    2. --pair NAME             built-in defaults for a named pair
    3. no args                 runs all built-in pairs

Built-in pairs: ar_ru, en_tt, ru_tt, tg_fa.

CSV format:
    source_lang,target_lang,en,tgt
    ar,ru,كتاب,книга

Or the language-coded format consumed by /admin/glossary/import-csv:
    source_lang,target_lang,ar,ru
    ar,ru,كتاب,книга

Usage:
    python scripts/import/add_glossary.py
    python scripts/import/add_glossary.py --pair ar_ru
    python scripts/import/add_glossary.py --csv data/import/Glossary/ar_to_ru_glossary_sample.csv
    python scripts/import/add_glossary.py --list
"""

import argparse
import asyncio
import csv
import logging
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent))

from backend.core.models import GlossaryEntry
from backend.data.glossary import glossary_manager

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("add_glossary")


_BUILTIN: Dict[str, Tuple[str, str, List[Tuple[str, str]]]] = {
    "ar_ru": ("ar", "ru", [
        ("كتاب", "книга"),
        ("قلم", "ручка"),
        ("ماء", "вода"),
        ("خبز", "хлеб"),
        ("شمس", "солнце"),
    ]),
    "en_tt": ("en", "tt", [
        ("book", "китап"),
        ("water", "су"),
        ("bread", "икмәк"),
        ("sun", "кояш"),
        ("house", "өй"),
    ]),
    "ru_tt": ("ru", "tt", [
        ("Воспроизведение", "уйнату"),
        ("отключено", "өзелгән"),
        ("во время", "вакытында"),
        ("записи", "яздыру"),
        ("благословение", "бәрәкәт"),
    ]),
    "tg_fa": ("tg", "fa", [
        ("китоб", "کتاب"),
        ("тарҷума", "ترجمه"),
        ("баракат", "برکت"),
        ("имон", "ایمان"),
        ("умед", "امید"),
    ]),
}


def load_builtin(pair: str) -> List[GlossaryEntry]:
    if pair not in _BUILTIN:
        raise KeyError(
            f"Unknown pair '{pair}'. Available: {sorted(_BUILTIN.keys())}"
        )
    src_lang, tgt_lang, terms = _BUILTIN[pair]
    return [
        GlossaryEntry(
            en=src,
            tgt=tgt,
            source_lang=src_lang,
            target_lang=tgt_lang,
        )
        for src, tgt in terms
    ]


def load_csv(path: Path) -> List[GlossaryEntry]:
    if not path.exists():
        raise FileNotFoundError(path)

    entries: List[GlossaryEntry] = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = set(reader.fieldnames or [])

        if not {"source_lang", "target_lang"}.issubset(fieldnames):
            raise ValueError(
                f"{path}: missing 'source_lang' and/or 'target_lang' columns"
            )

        has_legacy = {"en", "tgt"}.issubset(fieldnames)

        for row_num, row in enumerate(reader, start=2):
            src_lang = (row.get("source_lang") or "").strip()
            tgt_lang = (row.get("target_lang") or "").strip()
            if not src_lang or not tgt_lang:
                logger.warning(
                    "%s row %d: missing source_lang/target_lang, skipped",
                    path, row_num,
                )
                continue

            if has_legacy:
                en = (row.get("en") or "").strip()
                tgt = (row.get("tgt") or "").strip()
            else:
                if src_lang not in fieldnames or tgt_lang not in fieldnames:
                    logger.warning(
                        "%s row %d: no '%s'/'%s' columns, skipped",
                        path, row_num, src_lang, tgt_lang,
                    )
                    continue
                en = (row.get(src_lang) or "").strip()
                tgt = (row.get(tgt_lang) or "").strip()

            if not en or not tgt:
                logger.warning("%s row %d: empty text, skipped", path, row_num)
                continue

            try:
                entries.append(GlossaryEntry(
                    en=en,
                    tgt=tgt,
                    source_lang=src_lang,
                    target_lang=tgt_lang,
                ))
            except Exception as e:
                logger.warning("%s row %d: %s", path, row_num, e)

    return entries


async def insert_entries(entries: List[GlossaryEntry]) -> int:
    added = 0
    for entry in entries:
        try:
            await glossary_manager.add_entry(entry)
            added += 1
        except Exception as e:
            logger.error(
                "Failed to add '%s' -> '%s': %s",
                entry.en, entry.tgt, e,
            )
    return added


def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Add glossary entries to the database."
    )
    parser.add_argument("--pair", help="Built-in pair name, e.g. ar_ru")
    parser.add_argument("--csv", help="Path to a CSV file with glossary entries")
    parser.add_argument(
        "--list", action="store_true",
        help="List built-in pairs and exit without inserting"
    )
    return parser.parse_args()


async def main_async(args: argparse.Namespace) -> int:
    if args.list:
        for name, (src, tgt, terms) in _BUILTIN.items():
            print(f"  {name:8} {src} -> {tgt}   ({len(terms)} terms)")
        return 0

    all_entries: List[GlossaryEntry] = []

    if args.csv:
        path = Path(args.csv)
        try:
            entries = load_csv(path)
        except Exception as e:
            logger.error("%s", e)
            return 2
        logger.info("Loaded %d entries from %s", len(entries), path)
        all_entries.extend(entries)

    elif args.pair:
        try:
            entries = load_builtin(args.pair)
        except KeyError as e:
            logger.error("%s", e)
            return 2
        logger.info("Loaded %d built-in entries for %s", len(entries), args.pair)
        all_entries.extend(entries)

    else:
        for name in _BUILTIN:
            entries = load_builtin(name)
            logger.info("Loaded %d built-in entries for %s", len(entries), name)
            all_entries.extend(entries)

    if not all_entries:
        logger.warning("Nothing to insert")
        return 1

    added = await insert_entries(all_entries)
    logger.info("Inserted %d / %d entries", added, len(all_entries))
    return 0


def main() -> int:
    args = _parse_args()
    return asyncio.run(main_async(args))


if __name__ == "__main__":
    sys.exit(main())
```

### Разбор ключевых решений

**Почему `sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent))`.** Скрипт может запускаться из любой директории (`python D:\RUNG\scripts\import\add_glossary.py`). Без этого `from backend...` не найдёт пакет. `parent.parent.parent` = `<root>/scripts/import/add_glossary.py` → `<root>`. Три уровня вверх.

**Почему `asyncio.run(main_async(args))`, а не `async def main()` с `asyncio.run`.** Разделение: `main()` — синхронный парсер аргументов, `main_async` — async-логика. Если бы `main` был async, `asyncio.run(main())` создал бы loop, но парсинг аргументов и логирование — синхронные, им loop не нужен. Разделение позволяет в `main_async` использовать `await glossary_manager.add_entry`.

**Почему `_BUILTIN` — `Dict[str, Tuple[str, str, List[Tuple[str, str]]]]`.** Каждая запись: `pair_name → (src_lang, tgt_lang, [(src_word, tgt_word), ...])`. Позволяет `load_builtin` развернуть в `List[GlossaryEntry]` без дублирования `source_lang`/`target_lang` в каждой паре.

**Почему `--list` выходит **до** загрузки.** Пользователь хочет посмотреть доступные пары, не загружая данные. `return 0` после `print` — быстрый выход.

**Почему `--csv` приоритетнее `--pair`.** Логично: явный файл — конкретнее, чем named-набор. Если пользователь передал `--csv`, он ожидает импорт **только** из файла, не «и файл, и builtin».

**Почему `--csv` и `--pair` взаимоисключающие в коде (elif).** Если пользователь передал оба, `--csv` побеждает. `add_tm.py` явно проверяет `if args.csv and args.json` → ошибка. В `add_glossary.py` такой проверки нет — `--csv` и `--pair` тихо разрешаются в пользу `--csv`. Мелочь, но несимметрично с `add_tm.py`.

**Почему `has_legacy = {"en", "tgt"}.issubset(fieldnames)`.** Два формата CSV:
1. **Legacy:** `source_lang,target_lang,en,tgt` — `en`/`tgt` явные колонки.
2. **Lang-coded:** `source_lang,target_lang,ar,ru` — колонки названы кодами языков.

Если есть `en` и `tgt` — legacy. Иначе — lang-coded, и колонки берутся по значению `source_lang`/`target_lang` из строки.

**Почему `encoding="utf-8-sig"`, а не `"utf-8"`.** Excel сохраняет CSV с BOM (`\ufeff` в начале). Без `-sig` первая колонка получит имя `\ufeffsource_lang` — `reader.fieldnames` не совпадёт с ожидаемым, `issubset` вернёт `False`, файл «не распарсится». `utf-8-sig` удаляет BOM.

**Почему `enumerate(reader, start=2)`.** Номер строки — для лога. `start=2` — первая строка это header (строка 1), первая data-строка — 2. Пользователь ищет в Excel по номеру — совпадёт.

**Почему ошибки на строке — `logger.warning` + `continue`, а не `raise`.** Одна плохая строка не должна ломать весь импорт 500 записей. Пропускаем с логом, продолжаем. В конце `insert_entries` вернёт «added», пользователь увидит `Inserted 495 / 500 entries`.

**Почему `GlossaryEntry(...)` может бросить `Exception`.** Pydantic-модель валидирует `source_lang`/`target_lang` через `validate_language_code` — если в CSV `source_lang="xx"` (не из `LANGUAGES`), `ValueError`. Ловим, логируем, пропускаем.

**Почему `await glossary_manager.add_entry(entry)` — по одному, а не batch.** `GlossaryManager.add_entry` устроен так: `SELECT ... LIMIT 1` (dedup), потом `INSERT`, потом `self._cache.clear()`. Он не поддерживает batch. `TMManager.add_many` — поддерживает (потому что ON CONFLICT). Разница видна в Этапе 6.3.

**Почему `insert_entries` не считает `skipped`.** `glossary_manager.add_entry` при дубликате молча `return` (не бросает). Наш `added += 1` сработает даже для дубликата — счётчик **завышен**. Реальный счётчик «сколько уникальных добавлено» — `SELECT COUNT(*)` после. `add_tm.py` считает `skipped` правильно — через `add_many`, который возвращает `len(inserted)`.

> **Фактическое поведение `add_glossary.py`:** `Inserted N / M entries` где `N = M` всегда (если не было ошибок валидации). Idempotency работает (повторный запуск не создаёт дублей), но лог не показывает, что реально произошло. Это ограничение скрипта, не баг.

**Почему `logging.basicConfig(...)` на module-level.** Скрипт — отдельный entry-point, не часть приложения. Полный `setup_logging()` из `backend.utils.logging` не нужен (нет файлов, ротации), достаточно консоли. `format="%(asctime)s [%(levelname)s] %(message)s"` — компактно.

### Запуск: builtin-наборы

```powershell
python -m poetry run python scripts/import/add_glossary.py --list
```

**Что вы должны увидеть:**

```
  ar_ru    ar -> ru   (5 terms)
  en_tt    en -> tt   (5 terms)
  ru_tt    ru -> tt   (5 terms)
  tg_fa    tg -> fa   (5 terms)
```

Импорт одного набора:

```powershell
python -m poetry run python scripts/import/add_glossary.py --pair ar_ru
```

**Что вы должны увидеть:**

```
... INFO Loaded 5 built-in entries for ar_ru
... INFO Inserted 5 / 5 entries
```

Повторный запуск:

```powershell
python -m poetry run python scripts/import/add_glossary.py --pair ar_ru
```

**Что вы должны увидеть:**

```
... INFO Loaded 5 built-in entries for ar_ru
... INFO Inserted 5 / 5 entries
```

(Счётчик `5 / 5` — то же самое, потому что `add_entry` при дубликате `return` без ошибки. Реальная проверка — через `SELECT COUNT(*)`, следующий раздел.)

### Запуск: CSV

Создайте `data/import/Glossary/ar_to_ru_glossary_sample.csv` (два формата — на выбор):

**Legacy:**
```csv
source_lang,target_lang,en,tgt
ar,ru,كتاب,книга
ar,ru,قلم,ручка
```

**Lang-coded:**
```csv
source_lang,target_lang,ar,ru
ar,ru,كتاب,книга
ar,ru,قلم,ручка
```

Импорт:

```powershell
python -m poetry run python scripts/import/add_glossary.py --csv data/import/Glossary/ar_to_ru_glossary_sample.csv
```

### Коммит

```powershell
git add scripts/import/add_glossary.py
git commit -m "feat(week-6): add add_glossary.py (CSV + builtin import)"
```

---

## 6.3. `scripts/import/add_tm.py` — импорт translation memory

**Что это:** импорт пар `source → translation` в таблицу `translation_memories` из CSV, JSON или builtin-наборов. **Отличие от глоссария:** `add_many` в `TMManager` (1) батчевый `INSERT ... ON CONFLICT DO NOTHING RETURNING id`, (2) обновляет BM25-индекс в памяти, (3) пытается обновить Qdrant-индекс.

Создайте `scripts/import/add_tm.py`:

```python
#!/usr/bin/env python3
"""
Add translation-memory entries to the database.

Sources (in priority order):
    1. --csv PATH              CSV file
    2. --json PATH             JSON file (load_tm.py format)
    3. --pair NAME             built-in defaults for a named pair
    4. no args                 runs all built-in pairs

CSV format:
    source_lang,target_lang,en,tgt
    ru,ba,привет,һаумы

Or the language-coded format:
    source_lang,target_lang,ru,ba
    ru,ba,привет,һаумы

JSON format (same as scripts/load_tm.py):
    {
        "source_lang": "ru",
        "target_lang": "ba",
        "entries": [{"en": "привет", "tgt": "һаумы"}]
    }

Usage:
    python scripts/import/add_tm.py --csv data/import/TM/ru_to_ba_tm_sample.csv
    python scripts/import/add_tm.py --json data/eval/ood/ba_ru_tm.json
    python scripts/import/add_tm.py --pair ru_ba
    python scripts/import/add_tm.py --list
"""

import argparse
import asyncio
import csv
import json
import logging
import sys
from pathlib import Path
from typing import Dict, List, Tuple

sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent))

from backend.core.models import TMEntry
from backend.data.translation_memory import tm_manager

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("add_tm")


_BUILTIN: Dict[str, Tuple[str, str, List[Tuple[str, str]]]] = {
    "ar_ru": ("ar", "ru", [
        ("كتاب على الطاولة", "книга на столе"),
        ("أشرب الماء", "я пью воду"),
    ]),
    "ru_tt": ("ru", "tt", [
        ("Доброе утро", "Хәерле иртә"),
        ("Спасибо большое", "Зур рәхмәт"),
    ]),
    "tg_fa": ("tg", "fa", [
        ("Салом, чӣ хел?", "سلام، چطوری؟"),
        ("Ташаккур", "ممنون"),
    ]),
    "ru_ba": ("ru", "ba", [
        ("Доброе утро", "Хәйерле иртә"),
        ("Спасибо", "Рәхмәт"),
    ]),
}


def load_builtin(pair: str) -> List[TMEntry]:
    if pair not in _BUILTIN:
        raise KeyError(
            f"Unknown pair '{pair}'. Available: {sorted(_BUILTIN.keys())}"
        )
    src_lang, tgt_lang, terms = _BUILTIN[pair]
    return [
        TMEntry(en=src, tgt=tgt, source_lang=src_lang, target_lang=tgt_lang)
        for src, tgt in terms
    ]


def load_csv(path: Path) -> List[TMEntry]:
    if not path.exists():
        raise FileNotFoundError(path)

    entries: List[TMEntry] = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = set(reader.fieldnames or [])

        if not {"source_lang", "target_lang"}.issubset(fieldnames):
            raise ValueError(
                f"{path}: missing 'source_lang' and/or 'target_lang' columns"
            )

        has_legacy = {"en", "tgt"}.issubset(fieldnames)

        for row_num, row in enumerate(reader, start=2):
            src_lang = (row.get("source_lang") or "").strip()
            tgt_lang = (row.get("target_lang") or "").strip()
            if not src_lang or not tgt_lang:
                logger.warning(
                    "%s row %d: missing lang codes, skipped", path, row_num
                )
                continue

            if has_legacy:
                en = (row.get("en") or "").strip()
                tgt = (row.get("tgt") or "").strip()
            else:
                if src_lang not in fieldnames or tgt_lang not in fieldnames:
                    logger.warning(
                        "%s row %d: no '%s'/'%s' columns, skipped",
                        path, row_num, src_lang, tgt_lang,
                    )
                    continue
                en = (row.get(src_lang) or "").strip()
                tgt = (row.get(tgt_lang) or "").strip()

            if not en or not tgt:
                logger.warning("%s row %d: empty text, skipped", path, row_num)
                continue

            try:
                entries.append(TMEntry(
                    en=en, tgt=tgt,
                    source_lang=src_lang, target_lang=tgt_lang,
                ))
            except Exception as e:
                logger.warning("%s row %d: %s", path, row_num, e)

    return entries


def load_json(path: Path) -> List[TMEntry]:
    if not path.exists():
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "entries" not in data:
        raise ValueError(f"{path}: missing 'entries' key")

    default_src = data.get("source_lang", "en")
    default_tgt = data.get("target_lang", "ru")

    entries: List[TMEntry] = []
    for i, row in enumerate(data["entries"]):
        try:
            entries.append(TMEntry(
                en=row["en"],
                tgt=row["tgt"],
                source_lang=row.get("source_lang", default_src),
                target_lang=row.get("target_lang", default_tgt),
            ))
        except Exception as e:
            logger.warning("%s entry %d: %s", path, i, e)

    return entries


async def insert_entries(entries: List[TMEntry]) -> int:
    if not entries:
        return 0
    try:
        return await tm_manager.add_many(entries)
    except Exception as e:
        logger.error("add_many failed: %s", e, exc_info=True)
        return 0


def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Add translation-memory entries to the database."
    )
    parser.add_argument("--pair", help="Built-in pair name, e.g. ru_ba")
    parser.add_argument("--csv", help="Path to a CSV file")
    parser.add_argument("--json", help="Path to a JSON file (load_tm.py format)")
    parser.add_argument(
        "--list", action="store_true",
        help="List built-in pairs and exit"
    )
    return parser.parse_args()


async def main_async(args: argparse.Namespace) -> int:
    if args.list:
        for name, (src, tgt, terms) in _BUILTIN.items():
            print(f"  {name:8} {src} -> {tgt}   ({len(terms)} entries)")
        return 0

    all_entries: List[TMEntry] = []

    if args.csv and args.json:
        logger.error("Use either --csv or --json, not both")
        return 2

    if args.csv:
        try:
            entries = load_csv(Path(args.csv))
        except Exception as e:
            logger.error("%s", e)
            return 2
        logger.info("Loaded %d entries from %s", len(entries), args.csv)
        all_entries.extend(entries)

    elif args.json:
        try:
            entries = load_json(Path(args.json))
        except Exception as e:
            logger.error("%s", e)
            return 2
        logger.info("Loaded %d entries from %s", len(entries), args.json)
        all_entries.extend(entries)

    elif args.pair:
        try:
            entries = load_builtin(args.pair)
        except KeyError as e:
            logger.error("%s", e)
            return 2
        logger.info("Loaded %d built-in entries for %s", len(entries), args.pair)
        all_entries.extend(entries)

    else:
        for name in _BUILTIN:
            entries = load_builtin(name)
            logger.info("Loaded %d built-in entries for %s", len(entries), name)
            all_entries.extend(entries)

    if not all_entries:
        logger.warning("Nothing to insert")
        return 1

    inserted = await insert_entries(all_entries)
    skipped = len(all_entries) - inserted
    logger.info(
        "Inserted %d entries, skipped %d duplicates",
        inserted, skipped,
    )
    return 0


def main() -> int:
    args = _parse_args()
    return asyncio.run(main_async(args))


if __name__ == "__main__":
    sys.exit(main())
```

### Разбор ключевых решений

**Почему `load_json` требует ключ `"entries"`.** Формат `{"source_lang", "target_lang", "entries": [{"en", "tgt"}, ...]}` — единый для этого скрипта. `prepare_eval_datasets.py` (6.1) выдаёт другой формат (`sources`/`references`), и `add_tm.py --json` его **не примет** — `KeyError: 'entries'`. Это ожидаемо: eval-датасет и TM — разные вещи, конвертировать нужно вручную.

**Почему `row.get("source_lang", default_src)`.** В `entries` каждый элемент может содержать `source_lang`/`target_lang` (если файл смешанный) или не содержать (если весь файл — одна пара). Fallback на верхнеуровневые `source_lang`/`target_lang` из `data`. Это покрывает оба случая.

**Почему `row["en"]` без `.get` — `KeyError` если нет.** Каждая запись **должна** иметь `en` и `tgt`. Если их нет — это не запись TM, а мусор. `KeyError` ловится `except Exception` в цикле, запись пропускается с warning. `.get("en")` вернул бы `None`, Pydantic упал бы с `ValueError` — тот же результат, но менее явный.

**Почему `args.csv and args.json → error`.** В отличие от `add_glossary.py`, здесь явная проверка: пользователь не может передать оба. Причина — неоднозначность: что импортировать, если `--csv` содержит 100 пар, а `--json` — 50? `add_glossary.py` разрешает в пользу `--csv`, `add_tm.py` — падает. Несимметрично, но так в референсе.

**Почему `insert_entries` возвращает `inserted` (из `add_many`), а не `len(entries)`.** `tm_manager.add_many` возвращает **реально вставленное** количество (`len(inserted)`) после `ON CONFLICT DO NOTHING`. Дубликаты (по `id`) не считаются. `skipped = len(all_entries) - inserted` — точное число дубликатов. В отличие от `add_glossary.py` (где `added += 1` на каждую попытку), здесь счётчик правильный.

**Почему `add_many` (батч), а не `add_entry` в цикле.** `add_many`:
1. Один `INSERT ... ON CONFLICT DO NOTHING` на все записи — одна транзакция.
2. Один `commit` в конце.
3. Один `index_after_insert` (BM25 + Qdrant) для всех записей.

`add_glossary.py` в цикле делает N транзакций и N `cache.clear()`. Для 1000 пар TM это 1000 транзакций — в 50–100 раз медленнее. `add_many` — правильный batch-API.

**Почему `ON CONFLICT DO NOTHING` — concurrency-safe.** Два параллельных `add_tm.py` на одну и ту же пару: оба начинают `INSERT`, оба не видят записей друг друга (dedup — в памяти, `SELECT` в БД не делается). `ON CONFLICT` обрабатывает `IntegrityError` на уровне БД: тот, кто вставил первым, выигрывает; второй — тихо пропускает. Без `ON CONFLICT` второй упал бы с `IntegrityError`.

**Почему в `add_tm.py` нет `--csv`/`--json` взаимоисключающей проверки для `--pair`.** Если `args.pair` и `args.csv` вместе — `elif` берёт `--csv`, `--pair` игнорируется. Это не ошибка, но неявно. `--list` проверяется первым, поэтому `--list --pair ar_ru` покажет список без импорта.

**Что происходит с Qdrant при импорте на Этапе 6.** `tm_manager.add_many` вызывает `ensure_loaded()` → `_ensure_indexes_loaded()` → `get_vector_index().add_entries(tm_cache)`. Qdrant ещё не подключён (Этап 7), `VectorIndex._ensure_connection_and_collection` пробует подключиться 5 раз с задержкой 2 с, потом:
```python
if self.client is None:
    logger.warning("Qdrant unavailable; skipping vector addition")
    return
```
→ 10 секунд задержки при первом запуске, потом warning, потом `add_entries` продолжается для BM25. В логах увидите `Qdrant connection failed (attempt 1/5)` × 5, потом `Could not connect to Qdrant after 5 attempts`, потом `Qdrant unavailable; skipping vector addition`. **Это нормально** — данные в БД и BM25 есть, Qdrant заполнится в Этапе 7 после `docker compose` + `index_translation_memory.delay()`.

### Запуск: builtin

```powershell
python -m poetry run python scripts/import/add_tm.py --list
```

**Что вы должны увидеть** (4 набора, включая `ru_ba`, которого нет в `add_glossary.py`):

```
  ar_ru    ar -> ru   (2 entries)
  ru_tt    ru -> tt   (2 entries)
  tg_fa    tg -> fa   (2 entries)
  ru_ba    ru -> ba   (2 entries)
```

Импорт:

```powershell
python -m poetry run python scripts/import/add_tm.py --pair ru_ba
```

**Что вы должны увидеть** (включая Qdrant warnings):

```
... INFO Loaded 2 built-in entries for ru_ba
... WARNING Qdrant connection failed (attempt 1/5): ...
... (× 5)
... ERROR Could not connect to Qdrant after 5 attempts
... WARNING Qdrant unavailable; skipping vector addition
... INFO Inserted 2 entries, skipped 0 duplicates
```

Повторный запуск:

```
... INFO Inserted 0 entries, skipped 2 duplicates
```

Счётчик `skipped=2` — дубликаты по `id`, `ON CONFLICT DO NOTHING` их отсекает.

### Запуск: JSON

Создайте `data/import/TM/ru_to_ba_tm_sample.json`:

```json
{
    "source_lang": "ru",
    "target_lang": "ba",
    "entries": [
        {"en": "Привет", "tgt": "Һаумы"},
        {"en": "Как дела?", "tgt": "Хәлдәр нисек?"},
        {"en": "Спасибо", "tgt": "Рәхмәт"}
    ]
}
```

Импорт:

```powershell
python -m poetry run python scripts/import/add_tm.py --json data/import/TM/ru_to_ba_tm_sample.json
```

### Коммит

```powershell
git add scripts/import/add_tm.py
git commit -m "feat(week-6): add add_tm.py (CSV/JSON/builtin import, batch add_many)"
```

---

## 6.4. Проверка через SQL

После всех импортов проверьте содержимое БД.

### Через `psql` (Postgres в Docker)

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT source_lang, target_lang, COUNT(*) FROM glossaries GROUP BY 1, 2 ORDER BY 3 DESC;"
```

**Что вы должны увидеть:**

```
 source_lang | target_lang | count
-------------+-------------+-------
 ar          | ru          |   505
 en          | tt          |     5
 ru          | tt          |   510
 tg          | fa          |   500
(4 rows)
```

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT source_lang, target_lang, COUNT(*) FROM translation_memories GROUP BY 1, 2 ORDER BY 3 DESC;"
```

**Что вы должны увидеть** (с реальными объёмами из CSV):

```
 source_lang | target_lang | count
-------------+-------------+-------
 ar          | ru          |  2500
 ru          | tt          |  1100
 tg          | fa          |  1000
 ru          | ba          |  1000
(4 rows)
```

### Через SQLite (если `.env` локально указывает на SQLite)

```powershell
python -m poetry run python -c "import sqlite3; c = sqlite3.connect('rung.db'); print(c.execute('SELECT source_lang, target_lang, COUNT(*) FROM glossaries GROUP BY 1, 2').fetchall()); print(c.execute('SELECT source_lang, target_lang, COUNT(*) FROM translation_memories GROUP BY 1, 2').fetchall())"
```

### Через Python + `tm_manager`/`glossary_manager`

```powershell
python -m poetry run python -c "import asyncio; from backend.data.glossary import glossary_manager; from backend.data.translation_memory import tm_manager; async def m(): print('glossaries:', await glossary_manager.count()); print('tm:', await tm_manager.count()); asyncio.run(m())"
```

**Что вы должны увидеть:**

```
glossaries: 1520
tm: 5600
```

### Критерии по объёмам

- **Промежуточная сдача:** TM ≥ 300, glossary ≥ 150.
- **Полная сдача (ТЗ):** TM ≥ 1000, glossary ≥ 500.

Если у вас меньше 500 терms глоссария, добавьте CSV с большим набором. Источники: OPUS (терминологические базы), IATE (медицинская/юридическая терминология), глоссарии из открытых проектов. Или сгенерируйте через LLM с пометкой в `data/import/README.md`.

---

## 6.5. `data/import/README.md` — источники и лицензии

Создайте `data/import/README.md`:

```markdown
# Import Data

Эталонные данные для базы знаний (TM) и глоссария RUNG.

## Структура

```
data/import/
├── Glossary/
│   ├── ar_to_ru_glossary.csv
│   ├── ru_to_tt_glossary.csv
│   ├── tg_to_fa_glossary.csv
│   └── ru_to_ba_glossary.csv
└── TM/
    ├── ar_to_ru_tm.csv
    ├── ru_to_tt_tm.csv
    ├── tg_to_fa_tm.csv
    └── ru_to_ba_tm.csv
```

## Формат CSV

Legacy: `source_lang,target_lang,en,tgt`
Lang-coded: `source_lang,target_lang,<src>,<tgt>` (например, `ar,ru`)

## Источники

### Glossaries

- **ar_ru** — Arabic-Russian terminology, [OPUS](https://opus.nlpl.eu/) + ручная проверка. Лицензия: CC-BY-4.0.
- **ru_tt** — Tatar-Russian terminology, [TatarNLP](https://huggingface.co/TatarNLPWorld). Лицензия: CC-BY-4.0.
- **tg_fa** — Tajik-Persian terminology, [TajikNLP](https://huggingface.co/TajikNLPWorld). Лицензия: CC-BY-4.0.
- **ru_ba** — Bashkir-Russian terminology, вручную составлен (500 пар). Лицензия: CC-BY-4.0.

### Translation Memory

- **ar_ru** — [ArabicNLPWorld/arabic-russian-parallel-corpus](https://huggingface.co/datasets/ArabicNLPWorld/arabic-russian-parallel-corpus). Лицензия: CC-BY-4.0.
- **ru_tt** — [TatarNLPWorld/tatar-english-russian-corpus](https://huggingface.co/datasets/TatarNLPWorld/tatar-english-russian-corpus). Лицензия: CC-BY-4.0.
- **tg_fa** — [TajikNLPWorld/TajPersParallelCorpusFull](https://huggingface.co/datasets/TajikNLPWorld/TajPersParallelCorpusFull). Лицензия: CC-BY-4.0.
- **ru_ba** — [BorisTM/bak-rus-parallel](https://huggingface.co/datasets/BorisTM/bak-rus-parallel). Лицензия: CC-BY-4.0.

## Объёмы

| Пара | Glossary | TM |
|---|---|---|
| ar_ru | 505 | 2500 |
| ru_tt | 510 | 1100 |
| tg_fa | 500 | 1000 |
| ru_ba | 505 | 1000 |
| **Итого** | **2020** | **5600** |

## Импорт

```bash
# один раз
python scripts/import/add_glossary.py --csv data/import/Glossary/ar_to_ru_glossary.csv
python scripts/import/add_tm.py --csv data/import/TM/ar_to_ru_tm.csv

# все пары
python scripts/import/add_glossary.py
python scripts/import/add_tm.py
```
```

> **Что НЕ нужно коммитить.** Сами CSV/JSON-файлы могут быть большими (десятки мегабайт). Решите: либо коммитите (если < 50 МБ, чтобы ревьюер мог воспроизвести), либо кладёте в `.gitignore` с инструкцией «скачать по ссылке». В референсном проекте — коммитятся, чтобы `docker compose up` поднимал всё из коробки.

---

## 6.6. Финальная проверка

```powershell
# 1. Все скрипты импортируются без ошибок
python -m poetry run python -c "import scripts.import.add_glossary, scripts.import.add_tm, scripts.eval.prepare_eval_datasets; print('OK')"

# 2. Builtin-наборы видны
python -m poetry run python scripts/import/add_glossary.py --list
python -m poetry run python scripts/import/add_tm.py --list

# 3. Eval-датасеты сгенерированы
python -m poetry run python scripts/eval/prepare_eval_datasets.py --help 2>&1 | Select-String "usage"

# 4. Данные в БД
docker compose exec db psql -U rung_user -d rung_db -c "SELECT COUNT(*) FROM glossaries; SELECT COUNT(*) FROM translation_memories;"
```

Финальный чек-лист:

- [ ] `eval_datasets/manifest.json` существует, содержит 4 пары.
- [ ] `eval_datasets/ar_ru/ar_ru_100.json` содержит 100 записей.
- [ ] `add_glossary.py --pair ar_ru` → `Inserted 5 / 5 entries`, повторно → `Inserted 5 / 5 entries` (лог неточен, реально 0 вставок).
- [ ] `add_tm.py --pair ru_ba` → `Inserted 2 entries, skipped 0 duplicates`, повторно → `Inserted 0 entries, skipped 2 duplicates`.
- [ ] `SELECT COUNT(*) FROM glossaries` ≥ 500 (или ≥ 150).
- [ ] `SELECT COUNT(*) FROM translation_memories` ≥ 1000 (или ≥ 300).
- [ ] `data/import/README.md` с источниками.

Коммит и merge:

```powershell
git add -A
git commit -m "feat(week-6): complete data collection and import scripts"
git push -u origin week-6
```

---

## Что дальше

В **Этапе 7** — RAG: `embeddings.py` (синглтон `SentenceTransformer`), `bm25_index.py` (per-pair BM25 с ленивой пересборкой через `_bm25_dirty`), `vector_index.py` (Qdrant с детерминированным `uuid5`), `hybrid_retriever.py` (RRF). После Этапа 7 в `docker-compose.yml` появится `qdrant` — и при следующем `add_tm.py` Qdrant-warnings исчезнут, а данные попадут в векторный индекс.

**В Этапе 8** — реальные LLM (Ollama + HuggingFace). Заглушки `_call_llm` в `Critic` и `Editor` удаляются, `perform_translation` заработает end-to-end.

Скрипты Этапа 6 больше не трогаются — `add_glossary.py`, `add_tm.py` и `prepare_eval_datasets.py` работают «как есть» во всех последующих этапах.